# D4 – Huấn luyện, kiểm định chéo, hiệu chuẩn, ngưỡng

Đầu vào: `data/splits/train.csv`, `val_a.csv`, `val_b.csv` (đã chia sẵn bằng `src/split.py`, **không chia lại**).
Đầu ra: `models/` (TF-IDF, 3 mô hình, bộ hiệu chuẩn, `threshold.json`), `results/D4_cv.csv`,
`results/D4_hieu_chuan.csv`, `results/D4_tom_tat.txt`.

Quy tắc (README mục 9, RULES.md):
- **Không đọc `test.csv`** trong notebook này.
- Seed 42. Không in nội dung tin, không truy cập URL trong tin.
- TF-IDF chỉ fit trên dữ liệu huấn luyện (trong CV: chỉ trên các khối huấn luyện của fold).
- Siêu tham số chọn bằng CV 5 fold theo thời gian trong train; bộ hiệu chuẩn fit trên val_a;
  chọn bộ hiệu chuẩn và chốt ngưỡng trên val_b.

In [1]:
import json, os, pickle, platform, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from core import LABELS
from preprocess import fit_tfidf, parse_dates, transform_tfidf
from softmax import SoftmaxRegression, fit_path
from calibration import get_calibrator
from calibration_metrics import ece, log_loss

SEED = 42
np.random.seed(SEED)
K = len(LABELS)
T_START = time.time()
SPLITS, MODELS, RESULTS = ROOT / "data" / "splits", ROOT / "models", ROOT / "results"
MODELS.mkdir(exist_ok=True); RESULTS.mkdir(exist_ok=True)
pd.set_option("display.width", 200)

def load(name):
    assert name != "test", "D4 không được đọc test.csv"
    d = pd.read_csv(SPLITS / f"{name}.csv", encoding="utf-8-sig")
    d["y"] = d["label"].map({l: i for i, l in enumerate(LABELS)}).astype(int)
    d["_date"] = parse_dates(d["date"])
    return d

train, val_a, val_b = load("train"), load("val_a"), load("val_b")
print(pd.DataFrame({s: d.label.value_counts().reindex(LABELS) for s, d in
                    [("train", train), ("val_a", val_a), ("val_b", val_b)]}).T.assign(tong=lambda t: t.sum(axis=1)))
print("train: tin không date =", train._date.isna().sum())

label  lua_dao  rac  hop_le  tong
train      619  819     722  2160
val_a       55   70      60   185
val_b       43   69      60   172
train: tin không date = 495


## 2a. Kiểm định chéo theo thời gian trong train

Chia train thành 6 khối liên tiếp theo cùng logic phân tầng của `split.py`:
- trong từng lớp, các **nhóm** `nhom_gan_trung` có date được xếp theo date sớm nhất của nhóm (hòa thì theo id nhỏ nhất),
  rồi cắt thành 6 phần bằng nhau theo số tin; một nhóm luôn nằm trọn trong một khối;
- toàn bộ tin không date (SYN) thuộc khối 1 (đã kiểm tra: không nhóm nào lẫn tin có date và không date).

Fold k (k = 1..5): huấn luyện trên khối 1..k, đánh giá trên khối k+1.

In [2]:
N_BLOCKS = 6
g = train.groupby("nhom_gan_trung").agg(ngay=("_date", "min"), n=("id", "size"), id0=("id", "min"),
                                        label=("label", "first"), co_date=("_date", lambda s: s.notna().sum()))
assert (train.groupby("nhom_gan_trung").label.nunique() == 1).all()
mixed = train.groupby("nhom_gan_trung")._date.agg(lambda s: 0 < s.notna().sum() < len(s))
assert not mixed.any(), "có nhóm lẫn tin có date và không date"

khoi = {}
for lop, gl in g[g.ngay.notna()].groupby("label"):
    gl = gl.sort_values(["ngay", "id0"])
    N = gl.n.sum(); cum = gl.n.cumsum() - gl.n            # số tin của lớp xếp trước nhóm
    b = np.searchsorted(np.arange(1, N_BLOCKS) * N / N_BLOCKS, cum.to_numpy(), side="right") + 1
    khoi.update(dict(zip(gl.index, b)))
train["khoi"] = train.nhom_gan_trung.map(khoi).fillna(1).astype(int)   # không date -> khối 1

bang_khoi = pd.crosstab(train.label, train.khoi, margins=True).reindex(LABELS + ["All"])
print(bang_khoi)
print("\nTin có date theo khối:\n", pd.crosstab(train.label, train.khoi.where(train._date.notna())).reindex(LABELS))
assert (pd.crosstab(train.label, train.khoi) > 0).all().all(), "có khối thiếu lớp"
assert (train.groupby("nhom_gan_trung").khoi.nunique() == 1).all()
# thứ tự thời gian giữa các khối, trong từng lớp (theo date nhóm)
for lop in LABELS:
    d = train[(train.label == lop) & train._date.notna()]
    ng = d.groupby("nhom_gan_trung")._date.transform("min")
    mx = ng.groupby(d.khoi).agg(["min", "max"])
    assert (mx["min"].to_numpy()[1:] >= mx["max"].to_numpy()[:-1]).all(), lop

khoi       1    2    3    4    5    6   All
label                                      
lua_dao  238   76   76   77   76   76   619
rac      279  108  108  108  108  108   819
hop_le   257   96   90   93   93   93   722
All      774  280  274  278  277  277  2160

Tin có date theo khối:
 khoi     1.0  2.0  3.0  4.0  5.0  6.0
label                                
lua_dao   77   76   76   77   76   76
rac      109  108  108  108  108  108
hop_le    93   96   90   93   93   93


## 2b. Ba mô hình, lưới siêu tham số, CV

1. **Softmax tự viết** (`SoftmaxRegression`, solver `gd_armijo`), λ ∈ {1e-2, 3e-3, 1e-3, 3e-4, 1e-4},
   quét bằng `fit_path` (warm start từ λ lớn đến λ nhỏ). Mô hình nào `converged_ = False` sau
   `max_iter = 2000` thì **chạy tiếp** từ nghiệm hiện tại (warm start) thêm tối đa 2 × 8000 vòng —
   tương đương tăng `max_iter`; vẫn không hội tụ thì vẫn ghi vào bảng (`converged = 0`), không giấu.
   `n_iter` là tổng số vòng.
2. **Naive Bayes đa thức** (`MultinomialNB`), α ∈ {0.01, 0.03, 0.1, 0.3, 1}.
3. **SVM tuyến tính** (`LinearSVC`, one-vs-rest), C ∈ {0.01, 0.03, 0.1, 0.3, 1, 3}.
   SVM không có xác suất. **Trong CV**, xác suất của SVM = softmax(decision_function / T),
   với T fit (temperature scaling, `calibration.TemperatureCalibrator`) **trên chính các khối huấn luyện của fold**
   — không dùng khối đánh giá. Vì điểm trên dữ liệu huấn luyện thường "tự tin" hơn trên dữ liệu mới,
   log-loss CV của SVM có thể bị bi quan; cách này chỉ dùng để chọn C. Ở mô hình cuối, SVM được hiệu chuẩn trên val_a (2c).

Tiêu chí chọn: log-loss trung bình qua 5 fold (F1 macro theo argmax chỉ để báo cáo). std là độ lệch chuẩn mẫu (ddof = 1) qua 5 fold.

In [3]:
LAMS = [1e-2, 3e-3, 1e-3, 3e-4, 1e-4]
ALPHAS = [0.01, 0.03, 0.1, 0.3, 1]
CS = [0.01, 0.03, 0.1, 0.3, 1, 3]
EXTRA_ITER, EXTRA_ROUNDS = 8000, 2

def continue_until_converged(m, X, y):
    # Chạy tiếp (warm start) cho mô hình chưa hội tụ; trả về tổng số vòng.
    total = m.n_iter_
    for _ in range(EXTRA_ROUNDS):
        if m.converged_:
            break
        m.max_iter, m.warm_start = EXTRA_ITER, True
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            m.fit(X, y)
        total += m.n_iter_
    m.n_iter_total_ = total
    return m

def softmax_path(X, y, lams):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)   # chưa hội tụ -> xử lý bằng continue_until_converged
        ms = fit_path(X, y, lams, n_classes=K)
    return [continue_until_converged(m, X, y) for m in ms]

def svm_fit(C, X, y):
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always", ConvergenceWarning)
        m = LinearSVC(C=C, max_iter=20000, random_state=SEED).fit(X, y)
    m.n_conv_warn_ = sum(issubclass(x.category, ConvergenceWarning) for x in w)
    return m

def metrics(P, y):
    return log_loss(P, y), f1_score(y, P.argmax(1), average="macro", labels=list(range(K)))

rows = []
t_cv = time.time()
for k in range(1, N_BLOCKS):
    tr, ev = train[train.khoi <= k], train[train.khoi == k + 1]
    vw, vc = fit_tfidf(tr)                                   # chỉ fit trên khối huấn luyện
    Xtr, Xev = transform_tfidf(vw, vc, tr), transform_tfidf(vw, vc, ev)
    ytr, yev = tr.y.to_numpy(), ev.y.to_numpy()
    t0 = time.time()
    for m in softmax_path(Xtr, ytr, LAMS):
        ll, f1 = metrics(m.predict_proba(Xev), yev)
        rows.append(dict(mo_hinh="softmax", sieu_tham_so="lam", gia_tri=m.lam, fold=k, logloss=ll, f1_macro=f1,
                         n_iter=m.n_iter_total_, converged=int(m.converged_)))
    t1 = time.time()
    for a in ALPHAS:
        m = MultinomialNB(alpha=a).fit(Xtr, ytr)
        ll, f1 = metrics(m.predict_proba(Xev), yev)
        rows.append(dict(mo_hinh="naive_bayes", sieu_tham_so="alpha", gia_tri=a, fold=k, logloss=ll, f1_macro=f1))
    for C in CS:
        m = svm_fit(C, Xtr, ytr)
        cal = get_calibrator("temperature", K).fit(m.decision_function(Xtr), ytr)   # T fit trên khối huấn luyện
        ll, f1 = metrics(cal.predict_proba(m.decision_function(Xev)), yev)
        rows.append(dict(mo_hinh="svm", sieu_tham_so="C", gia_tri=C, fold=k, logloss=ll, f1_macro=f1,
                         svm_conv_warn=m.n_conv_warn_, T_fold=cal.T_))
    print(f"fold {k}: train {Xtr.shape[0]} tin, eval {Xev.shape[0]} tin, d = {Xtr.shape[1]}, "
          f"softmax {t1 - t0:.1f} s, NB+SVM {time.time() - t1:.1f} s")
cv_folds = pd.DataFrame(rows)
print(f"CV: {time.time() - t_cv:.1f} s")

fold 1: train 774 tin, eval 280 tin, d = 8378, softmax 3.1 s, NB+SVM 0.2 s


fold 2: train 1054 tin, eval 274 tin, d = 11155, softmax 4.8 s, NB+SVM 0.2 s


fold 3: train 1328 tin, eval 278 tin, d = 13243, softmax 8.0 s, NB+SVM 0.3 s


fold 4: train 1606 tin, eval 277 tin, d = 15592, softmax 11.7 s, NB+SVM 0.5 s


fold 5: train 1883 tin, eval 277 tin, d = 18029, softmax 13.4 s, NB+SVM 0.5 s
CV: 45.2 s


In [4]:
agg = (cv_folds.groupby(["mo_hinh", "sieu_tham_so", "gia_tri"], sort=False)
       .agg(logloss_mean=("logloss", "mean"), logloss_std=("logloss", "std"),
            f1_macro_mean=("f1_macro", "mean"), f1_macro_std=("f1_macro", "std"),
            n_iter=("n_iter", "mean"), n_iter_max=("n_iter", "max"),
            converged=("converged", "min"), so_fold_hoi_tu=("converged", "sum"))
       .reset_index())
agg.loc[agg.mo_hinh != "softmax", "so_fold_hoi_tu"] = np.nan   # chỉ có nghĩa với softmax
for c in ["n_iter", "n_iter_max", "converged", "so_fold_hoi_tu"]:
    agg[c] = agg[c].round().astype("Int64")            # NA cho NB, SVM
agg["chon"] = 0
best = agg.groupby("mo_hinh").logloss_mean.idxmin()
agg.loc[best, "chon"] = 1
agg.to_csv(RESULTS / "D4_cv.csv", index=False, encoding="utf-8-sig")
CHOSEN = {r.mo_hinh: r.gia_tri for r in agg[agg.chon == 1].itertuples()}
print(agg.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("\nĐã chọn:", CHOSEN)
n_nc = (cv_folds.query("mo_hinh == 'softmax'").converged == 0).sum()
print("Softmax chưa hội tụ (số (fold, λ)):", n_nc,
      "| SVM có ConvergenceWarning (số (fold, C)):", int((cv_folds.svm_conv_warn.fillna(0) > 0).sum()))

    mo_hinh sieu_tham_so  gia_tri  logloss_mean  logloss_std  f1_macro_mean  f1_macro_std  n_iter  n_iter_max  converged  so_fold_hoi_tu  chon
    softmax          lam     0.01        0.6717      0.05059         0.8735       0.03236     128         132          1               5     0
    softmax          lam    0.003        0.4899      0.05834         0.8928       0.02227     267         291          1               5     0
    softmax          lam    0.001        0.3772      0.05844         0.8964       0.02882     450         516          1               5     0
    softmax          lam   0.0003        0.3069      0.05614         0.8937       0.03262     683         873          1               5     0
    softmax          lam   0.0001        0.2765      0.05447         0.8882       0.03652     832        1120          1               5     1
naive_bayes        alpha     0.01        0.7328       0.2267         0.8739       0.03316    <NA>        <NA>       <NA>            <NA>     0

## 2c. Mô hình cuối và hiệu chuẩn

- TF-IDF fit trên **toàn bộ train**; 3 mô hình huấn luyện trên toàn bộ train với cấu hình đã chọn
  (softmax: `fit_path` từ λ lớn nhất xuống λ đã chọn, warm start).
- Điểm đưa vào bộ hiệu chuẩn (RULES.md): softmax → `decision_function`; NB → `predict_log_proba`; SVM → `decision_function`.
- Bộ hiệu chuẩn (`src/calibration.py`) **chỉ fit trên val_a**: Platt, PAVA (isotonic), temperature;
  softmax và NB thêm "none" (= softmax của điểm, tức xác suất gốc); SVM bắt buộc hiệu chuẩn nên không có "none".
- Chọn trên val_b: log-loss nhỏ nhất; các phương pháp có log-loss trong khoảng 0,005 so với nhỏ nhất
  thì chọn phương pháp có ECE top-label (M = 15) nhỏ hơn.

In [5]:
t_fin = time.time()
vec_word, vec_char = fit_tfidf(train)
X_train = transform_tfidf(vec_word, vec_char, train)
X_va, X_vb = transform_tfidf(vec_word, vec_char, val_a), transform_tfidf(vec_word, vec_char, val_b)
y_train, y_va, y_vb = train.y.to_numpy(), val_a.y.to_numpy(), val_b.y.to_numpy()
print("X_train", X_train.shape, "nnz", X_train.nnz)

lam_star = CHOSEN["softmax"]
path = softmax_path(X_train, y_train, [l for l in LAMS if l >= lam_star])
sm = path[-1]; assert sm.lam == lam_star
nb = MultinomialNB(alpha=CHOSEN["naive_bayes"]).fit(X_train, y_train)
svm = svm_fit(CHOSEN["svm"], X_train, y_train)
print(f"softmax λ = {sm.lam:g}: n_iter = {sm.n_iter_total_}, converged = {sm.converged_}")
print(f"NB α = {nb.alpha:g} | SVM C = {svm.C:g}, ConvergenceWarning = {svm.n_conv_warn_}")

for name, obj in [("tfidf_word", vec_word), ("tfidf_char", vec_char), ("naive_bayes", nb), ("svm", svm)]:
    with open(MODELS / f"{name}.pkl", "wb") as f:
        pickle.dump(obj, f)
sm.save(MODELS / "softmax.npz")
assert np.allclose(SoftmaxRegression.load(MODELS / "softmax.npz").decision_function(X_vb), sm.decision_function(X_vb))

def scores(model_name, X):
    # Điểm (n, K) đưa vào bộ hiệu chuẩn, theo RULES.md.
    return {"softmax": sm.decision_function, "naive_bayes": nb.predict_log_proba,
            "svm": svm.decision_function}[model_name](X)

MODEL_NAMES = ["softmax", "naive_bayes", "svm"]
S_va = {m: scores(m, X_va) for m in MODEL_NAMES}
S_vb = {m: scores(m, X_vb) for m in MODEL_NAMES}
print(f"Huấn luyện mô hình cuối: {time.time() - t_fin:.1f} s")

X_train (2160, 20586) nnz 789964


softmax λ = 0.0001: n_iter = 1230, converged = True
NB α = 1 | SVM C = 0.03, ConvergenceWarning = 0
Huấn luyện mô hình cuối: 18.3 s


In [6]:
METHODS = {"softmax": ["none", "platt", "isotonic", "temperature"],
           "naive_bayes": ["none", "platt", "isotonic", "temperature"],
           "svm": ["platt", "isotonic", "temperature"]}
TIE = 0.005
cal_rows, fitted = [], {}
for m in MODEL_NAMES:
    for meth in METHODS[m]:
        cal = get_calibrator(meth, K).fit(S_va[m], y_va)        # CHỈ fit trên val_a
        P = cal.predict_proba(S_vb[m])
        fitted[(m, meth)] = cal
        cal_rows.append(dict(mo_hinh=m, phuong_phap=meth, logloss_val_b=log_loss(P, y_vb),
                             ece_top_val_b=ece(P, y_vb, n_bins=15, kind="top_label"),
                             ece_lua_dao_val_b=ece(P[:, 0], (y_vb == 0).astype(int), n_bins=15, kind="binary"),
                             f1_macro_argmax_val_b=f1_score(y_vb, P.argmax(1), average="macro", labels=list(range(K)))))
hc = pd.DataFrame(cal_rows)
hc["chon"] = 0
for m, d in hc.groupby("mo_hinh"):
    cand = d[d.logloss_val_b <= d.logloss_val_b.min() + TIE]
    hc.loc[cand.ece_top_val_b.idxmin(), "chon"] = 1
hc.to_csv(RESULTS / "D4_hieu_chuan.csv", index=False, encoding="utf-8-sig")
CAL_CHOSEN = {r.mo_hinh: r.phuong_phap for r in hc[hc.chon == 1].itertuples()}
calibrators = {m: fitted[(m, meth)] for m, meth in CAL_CHOSEN.items()}
with open(MODELS / "calibrators.pkl", "wb") as f:
    pickle.dump(calibrators, f)
P_vb = {m: calibrators[m].predict_proba(S_vb[m]) for m in MODEL_NAMES}   # xác suất ĐÃ hiệu chuẩn trên val_b
print(hc.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print("\nBộ hiệu chuẩn được chọn:", CAL_CHOSEN)

    mo_hinh phuong_phap  logloss_val_b  ece_top_val_b  ece_lua_dao_val_b  f1_macro_argmax_val_b  chon
    softmax        none         0.2621         0.0599             0.0640                 0.8954     0
    softmax       platt         0.2642         0.0622             0.0675                 0.8857     0
    softmax    isotonic         0.4244         0.0398             0.0449                 0.8915     0
    softmax temperature         0.2387         0.0445             0.0353                 0.8954     1
naive_bayes        none         0.2968         0.0526             0.0659                 0.8893     1
naive_bayes       platt         0.3019         0.0818             0.0879                 0.8840     0
naive_bayes    isotonic         0.3787         0.0284             0.0281                 0.8963     0
naive_bayes temperature         0.2977         0.0556             0.0833                 0.8893     0
        svm       platt         0.2882         0.0611             0.0936          

## 2d. Ma trận chi phí và ngưỡng tại FPR = 1%

RULES.md chưa có ma trận chi phí cụ thể, nên dùng ma trận đề xuất trong `docs/ma_tran_chi_phi.md`
(`C[lớp thật, lớp đoán]`, thứ tự `LABELS` = lua_dao, rac, hop_le). Quy tắc quyết định:
ŷ(x) = argmin_j Σ_i C[i, j]·p(i|x), tức argmin theo cột của `P @ C`.

Ngưỡng: điểm = p(lua_dao | x) **sau hiệu chuẩn**, bài toán lừa đảo vs phần còn lại, báo lừa đảo khi điểm ≥ t.
Với mỗi mô hình, t = giá trị nhỏ nhất trong tập ứng viên (các điểm khác nhau trên val_b, thêm +∞) sao cho
FPR trên val_b ≤ 1%. val_b chỉ có 129 tin không lừa đảo, nên 1% ≈ 1,3 tin: cho phép tối đa **1** tin báo nhầm —
độ phân giải của ngưỡng rất thô (mỗi tin báo nhầm = 0,78 điểm %).

In [7]:
C_COST = np.array([[0, 5, 10],     # thật lua_dao: đoán lua_dao, rac, hop_le
                   [1, 0, 1],      # thật rac
                   [5, 3, 0]],     # thật hop_le
                  dtype=float)
MAX_FPR = 0.01
pi_train = (train.y.value_counts(normalize=True).reindex(range(K)).to_numpy())

def threshold_at_fpr(score, is_pos, max_fpr=MAX_FPR):
    # Ngưỡng nhỏ nhất (trong các điểm khác nhau + inf) sao cho FPR = P(score >= t | âm) <= max_fpr.
    neg, pos = score[~is_pos], score[is_pos]
    for t in np.append(np.unique(score), np.inf):
        fpr = float(np.mean(neg >= t))
        if fpr <= max_fpr:
            return float(t), fpr, float(np.mean(pos >= t)), int(np.sum(neg >= t))

is_fraud = y_vb == 0
thr = {}
for m in MODEL_NAMES:
    t, fpr, rec, n_fp = threshold_at_fpr(P_vb[m][:, 0], is_fraud)
    thr[m] = {"calibrator": CAL_CHOSEN[m], "threshold": None if np.isinf(t) else t,
              "fpr_val_b": fpr, "recall_lua_dao_val_b": rec, "so_bao_nham_val_b": n_fp}
threshold = {
    "labels": LABELS,
    "cost_matrix": C_COST.tolist(),
    "cost_matrix_note": "C[lớp thật, lớp đoán], thứ tự labels; xem docs/ma_tran_chi_phi.md",
    "fpr_muc_tieu": MAX_FPR,
    "diem": "p(lua_dao | x) sau hiệu chuẩn; báo lừa đảo khi điểm >= threshold",
    "n_lua_dao_val_b": int(is_fraud.sum()), "n_khong_lua_dao_val_b": int((~is_fraud).sum()),
    "pi_train": pi_train.tolist(),
    "models": thr,
}
with open(MODELS / "threshold.json", "w", encoding="utf-8") as f:
    json.dump(threshold, f, ensure_ascii=False, indent=2)
print(json.dumps(threshold, ensure_ascii=False, indent=2))

{
  "labels": [
    "lua_dao",
    "rac",
    "hop_le"
  ],
  "cost_matrix": [
    [
      0.0,
      5.0,
      10.0
    ],
    [
      1.0,
      0.0,
      1.0
    ],
    [
      5.0,
      3.0,
      0.0
    ]
  ],
  "cost_matrix_note": "C[lớp thật, lớp đoán], thứ tự labels; xem docs/ma_tran_chi_phi.md",
  "fpr_muc_tieu": 0.01,
  "diem": "p(lua_dao | x) sau hiệu chuẩn; báo lừa đảo khi điểm >= threshold",
  "n_lua_dao_val_b": 43,
  "n_khong_lua_dao_val_b": 129,
  "pi_train": [
    0.2865740740740741,
    0.37916666666666665,
    0.33425925925925926
  ],
  "models": {
    "softmax": {
      "calibrator": "temperature",
      "threshold": 0.7076425458887482,
      "fpr_val_b": 0.007751937984496124,
      "recall_lua_dao_val_b": 0.7906976744186046,
      "so_bao_nham_val_b": 1
    },
    "naive_bayes": {
      "calibrator": "none",
      "threshold": 0.9038263562879011,
      "fpr_val_b": 0.007751937984496124,
      "recall_lua_dao_val_b": 0.18604651162790697,
      "so_bao_nham_val_b"

## 2e. Tóm tắt cho báo cáo (lưu `results/D4_tom_tat.txt`)

In [8]:
def machine_info():
    ram = None
    try:
        import psutil
        ram = psutil.virtual_memory().total
    except ImportError:
        if os.name == "nt":
            import ctypes
            class MS(ctypes.Structure):
                _fields_ = [("dwLength", ctypes.c_ulong), ("dwMemoryLoad", ctypes.c_ulong),
                            ("ullTotalPhys", ctypes.c_ulonglong), ("ullAvailPhys", ctypes.c_ulonglong),
                            ("ullTotalPageFile", ctypes.c_ulonglong), ("ullAvailPageFile", ctypes.c_ulonglong),
                            ("ullTotalVirtual", ctypes.c_ulonglong), ("ullAvailVirtual", ctypes.c_ulonglong),
                            ("sullAvailExtendedVirtual", ctypes.c_ulonglong)]
            ms = MS(); ms.dwLength = ctypes.sizeof(MS)
            ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(ms))
            ram = ms.ullTotalPhys
    ram_s = f"{ram / 2**30:.1f} GB" if ram else "không xác định"
    return (f"CPU: {platform.processor() or platform.machine()} ({os.cpu_count()} luồng logic)\n"
            f"RAM: {ram_s}\nOS: {platform.platform()}\n"
            f"Python {platform.python_version()} | numpy {np.__version__} | "
            f"scikit-learn {sklearn.__version__} | pandas {pd.__version__}")

out = []
p = out.append
fmt = lambda v: f"{v:.4f}"
p("=== D4_cv: cấu hình được chọn (mean ± std qua 5 fold) ===")
p(agg[agg.chon == 1].to_string(index=False, float_format=fmt))
p("\n=== D4_hieu_chuan (val_b; bộ hiệu chuẩn fit trên val_a) ===")
p(hc.to_string(index=False, float_format=fmt))
p("\n=== val_b, sau hiệu chuẩn, quy tắc chi phí tối thiểu argmin_j Σ_i C[i,j]·p(i|x) ===")
for m in MODEL_NAMES:
    P = P_vb[m]
    E = P @ C_COST                                    # (n, K): chi phí kỳ vọng nếu đoán j
    pred = E.argmin(1)
    cm = confusion_matrix(y_vb, pred, labels=list(range(K)))
    p(f"\n[{m}] hiệu chuẩn = {CAL_CHOSEN[m]}")
    p(f"  F1 macro = {f1_score(y_vb, pred, average='macro', labels=list(range(K))):.4f}")
    p(f"  Chi phí trung bình thực tế mean C[y, ŷ] = {C_COST[y_vb, pred].mean():.4f}")
    p(f"  Chi phí kỳ vọng trung bình theo mô hình mean min_j (P·C)_j = {E.min(1).mean():.4f}")
    p("  Ma trận nhầm lẫn (hàng = thật, cột = đoán; " + ", ".join(LABELS) + "):")
    p(pd.DataFrame(cm, index=LABELS, columns=LABELS).to_string())
p("\n=== threshold.json ===")
p(json.dumps(threshold, ensure_ascii=False, indent=2))
p("\n=== Cấu hình máy ===")
p(machine_info())
p(f"Tổng thời gian chạy notebook: {time.time() - T_START:.1f} s")
summary = "\n".join(out)
(RESULTS / "D4_tom_tat.txt").write_text(summary, encoding="utf-8")
print(summary)

=== D4_cv: cấu hình được chọn (mean ± std qua 5 fold) ===
    mo_hinh sieu_tham_so  gia_tri  logloss_mean  logloss_std  f1_macro_mean  f1_macro_std  n_iter  n_iter_max  converged  so_fold_hoi_tu  chon
    softmax          lam   0.0001        0.2765       0.0545         0.8882        0.0365     832        1120          1               5     1
naive_bayes        alpha   1.0000        0.3656       0.0534         0.8498        0.0237    <NA>        <NA>       <NA>            <NA>     1
        svm            C   0.0300        0.3383       0.1173         0.8839        0.0296    <NA>        <NA>       <NA>            <NA>     1

=== D4_hieu_chuan (val_b; bộ hiệu chuẩn fit trên val_a) ===
    mo_hinh phuong_phap  logloss_val_b  ece_top_val_b  ece_lua_dao_val_b  f1_macro_argmax_val_b  chon
    softmax        none         0.2621         0.0599             0.0640                 0.8954     0
    softmax       platt         0.2642         0.0622             0.0675                 0.8857     0
   